read_flat_yml(input_file): read file with frequencies, like 

```
vorto1: 333 
vorto2: 22 
... 
vorto99: 1
```


In [4]:
# Defines read_flat_yml helper: parses a custom "word: frequency" file into a dict,
# avoiding yaml library's bug that converts string "false" to boolean False.
def read_flat_yml(input_file):
    with open(input_file, "r") as f:
        lines = f.readlines()

    all_data = {}
    for line in lines:
        vorto, kvanto = line.strip().split(": ")
        all_data[vorto] = int(kvanto)

    return all_data

In [5]:
# NOTE: shows why the standard yaml library is NOT used —
# it incorrectly converts the string "false" to boolean False.

# read with yaml library
# it has error: value 'false' is transformed to boolean False, instead to be just string
# don't use it

# import yaml

# file_name = "./data/source1_all_words.yaml"
# with open(file_name, 'r') as file:
#     data = yaml.safe_load(file)

# print(data.get("false", "string 'false' Not found"))
# print(data.get(False,  "bool 'False' Not found"))

# print("Type:", type(data))
# print("Length:", len(data))
# print("First 10 items:", list(data.items())[:10])


In [6]:
# Loads the main word-frequency dataset from source1_all_words.yaml into `data` dict
# and prints its type, total size, and a sample of the first 10 entries.
file_name = "./data/source1_all_words.yaml"

# data - dictionary with key as word and value as its frequency
data = read_flat_yml(file_name)

print("Type:", type(data))
print("Length:", len(data))
print("First 10 items:", list(data.items())[:10])


Type: <class 'dict'>
Length: 49464
First 10 items: [('la', 612479), ('de', 308999), ('kaj', 219359), ('en', 146802), ('al', 96398), ('estas', 71030), ('ne', 68672), ('mi', 57988), ('por', 57813), ('li', 56887)]


In [7]:
# Defines write_groups_to_yaml helper: writes grouped word lists to a YAML file,
# sorting each group by descending frequency then alphabetically (Esperanto locale).
import locale

def write_groups_to_yaml(groups, output_file):
    locale.setlocale(locale.LC_COLLATE, "eo.UTF-8")

    with open(output_file, "w") as f:
        for group_key, words in groups.items():
            f.write(f"\n{group_key} {len(words)}:\n")
            sorted_words = sorted(
                words,
                key=lambda item: (-int(item[1]), locale.strxfrm(item[0])),
            )
            for word, freq in sorted_words:
                f.write(f"\t{word}: {freq}\n")


# dict = {"a": [("ba", 333), ("ca", 2)], "u": [], "o": [("bo", 1)]}
# write_groups_to_yaml(dict, "./data/test.yaml")


In [8]:
# Groups all words by their grammatical ending (e.g. -as, -oj, -a, …);
# words with no matching ending or length ≤ 3 go into "other". Saves to groups_by_ending.yaml.

# calculate groups by ending

endings = [
    "as",
    "is",
    "os",
    "us",
    "ajn",
    "ojn",
    "ujn",
    "aj",
    "oj",
    "uj",
    "ej",
    "an",
    "on",
    "en",
    "in",
    "un",
    "aŭ",
    "eŭ",
    "a",
    "i",
    "e",
    "o",
    "u",
]
groups_by_endings = {ending: [] for ending in endings}
groups_by_endings["other"] = []
# groups_by_endings - dict with keys as endings and values as list of tuples (word, freq)

for word, freq in data.items():
    if len(word) <= 3:
        groups_by_endings["other"].append((word, freq))
        continue
    has_suffix = False
    for ending in endings:
        if word.endswith(ending):
            groups_by_endings[ending].append((word, freq))
            has_suffix = True
            break
    if not has_suffix:
        groups_by_endings["other"].append((word, freq))

# print groups into yaml file
output_file = "./data/groups_by_ending.yaml"
write_groups_to_yaml(groups_by_endings, output_file)
print(f"all groups by ending saved to {output_file}")

all groups by ending saved to ./data/groups_by_ending.yaml


In [9]:
# Exports the "other" group (words with unrecognized endings) to a separate CSV file
# other_endings.csv for manual review.

# print 'others' group to separate file

output_file = "./data/other_endings.csv"
with open(output_file, "w") as f:
    for word, freq in groups_by_endings.get("other", []):
        f.write(f"{word},{freq}\n")
print(f"Data with ending 'other' saved to {output_file}")


Data with ending 'other' saved to ./data/other_endings.csv


In [10]:
# Calculates per-ending statistics (unique word count + total frequency sum)
# and writes them to stats_by_endings.csv.

# calculate stats for each ending

output_file = "./data/stats_by_endings.csv"
with open(output_file, "w") as f:
    for ending, words in groups_by_endings.items():
        total_count = sum([freq for _, freq in words])
        f.write(f"{ending},{len(words)}, {total_count}\n")
print(f"ending stats saved to {output_file}")

# verbs: 3786
# nouns: 12815
# adjectives: 6674
# adverbs: 2976
# other: 326
# (...aŭ, numbers, prepositions, help and table words)


ending stats saved to ./data/stats_by_endings.csv


In [11]:
# Defines merge_endings_into_one helper: collapses multiple inflected-form groups
# into one canonical base form (e.g. lavas/lavis/lavos → lavi), summing frequencies.

def merge_endings_into_one(groups, endings, target):
    merged_data = {}
    for ending in endings:
        for word, freq in groups.get(ending, []):
            base_word = word[:-len(ending)] + target
            if base_word in merged_data:
                merged_data[base_word] += freq
            else:
                merged_data[base_word] = freq
        del groups[ending]
    return merged_data


In [12]:
# Merges all inflected forms into 5 canonical word-class groups (verbs→-i, nouns→-o,
# adjectives→-a, adverbs→-e, "who"-words→-u) and saves them to YAML files.
import copy

groups_by_endings_copy = copy.deepcopy(groups_by_endings)

verbs = merge_endings_into_one(groups_by_endings_copy, ["as", "os", "us", "is", "u", "i"], "i")
nouns = merge_endings_into_one(groups_by_endings_copy, ["ojn", "oj", "on", "o"], "o")
adjectives = merge_endings_into_one(groups_by_endings_copy, ["ajn", "aj", "an", "a"], "a")
adverbs = merge_endings_into_one(groups_by_endings_copy, ["en", "e"], "e")
who = merge_endings_into_one(groups_by_endings_copy, ["ujn", "uj", "un"], "u")


# print groups into yaml file
output_file = "./data/groups_by_ending__rest.yaml"
write_groups_to_yaml(groups_by_endings_copy, output_file)
print(f"rest of groups by ending saved to {output_file}")

five_groups = {"verbs": verbs.items(), "nouns": nouns.items(), "adjectives": adjectives.items(), "adverbs": adverbs.items(), "who": who.items()}
output_file = "./data/five_groups_by_ending.yaml"
write_groups_to_yaml(five_groups, output_file)
print(f"five groups by ending saved to {output_file}")


# rest of groups by ending saved to ./data/groups_by_ending__rest.yaml
# five groups by ending saved to ./data/five_groups_by_ending.yaml


rest of groups by ending saved to ./data/groups_by_ending__rest.yaml
five groups by ending saved to ./data/five_groups_by_ending.yaml


In [13]:
# Planned next step (not yet implemented): group words by derivational suffixes
# (-aĉ-, -ad-, -ig-, -iĝ-, -ul-, etc.).
#  group by suffixes


In [14]:
# Scratchpad / throwaway test cell: demonstrates Python dict deletion — no analytical purpose.

# this is Scratchpad, test cell

dict = {"a": 1, "b": 2, "c": 3}
print(dict)

del dict["a"]

print(dict)

print(type({}))


{'a': 1, 'b': 2, 'c': 3}
{'b': 2, 'c': 3}
<class 'dict'>
